<a href="https://colab.research.google.com/github/yashwanthks78/pytorch/blob/main/Simple_RNN_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [79]:
import torch
import torch.nn as nn
import torch.optim as optim

In [80]:


corpus = [
    "I love machine learning",
    "word2vec is great algorithm",
    "Implementing word2vec is fun"
]

In [81]:

sentences = [s.lower().split() for s in corpus]

In [82]:
sentences

[['i', 'love', 'machine', 'learning'],
 ['word2vec', 'is', 'great', 'algorithm'],
 ['implementing', 'word2vec', 'is', 'fun']]

In [83]:
vocab = sorted(set(word for sent in sentences for word in sent))

In [84]:
vocab

['algorithm',
 'fun',
 'great',
 'i',
 'implementing',
 'is',
 'learning',
 'love',
 'machine',
 'word2vec']

In [85]:
word2idx = {w:i for i,w in enumerate(vocab)}
# idx2word = {i:w for w,i in word2idx.items()}
idx2word = {i:w for i,w in enumerate(vocab)}

In [86]:
idx2word

{0: 'algorithm',
 1: 'fun',
 2: 'great',
 3: 'i',
 4: 'implementing',
 5: 'is',
 6: 'learning',
 7: 'love',
 8: 'machine',
 9: 'word2vec'}

In [87]:
vocab_size = len(vocab)

In [88]:
word2idx

{'algorithm': 0,
 'fun': 1,
 'great': 2,
 'i': 3,
 'implementing': 4,
 'is': 5,
 'learning': 6,
 'love': 7,
 'machine': 8,
 'word2vec': 9}

In [89]:
X = []
Y = []
for sent in sentences:
    for i in range(len(sent)-1):
        X.append(word2idx[sent[i]])
        Y.append(word2idx[sent[i+1]])

In [90]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [91]:
print(X)
print(Y)

tensor([3, 7, 8, 9, 5, 2, 4, 9, 5])
tensor([7, 8, 6, 5, 2, 0, 9, 5, 1])


In [92]:
XX = X.reshape(3,3)

In [93]:
YY = Y.reshape(3,3)

In [94]:
vocab_size

10

In [105]:

class NextWordRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x) # [3, 3, 100]
        out, hx = self.rnn(x) # [3, 3, 16], [1, 3, 16]
        out = out[:, -1, :] # [3, 16]
        out = self.fc(out) # [3, 10]
        return out


In [96]:
model = NextWordRNN(10)

In [97]:
outputs = model(XX)

In [100]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [106]:

for epoch in range(300):
    optimizer.zero_grad()
    outputs = model(X)
    loss = loss_fn(outputs, Y)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

IndexError: too many indices for tensor of dimension 2

In [102]:
def predict_next(word):
    model.eval()
    idx = torch.tensor([[word2idx[word.lower()]]])

    with torch.no_grad():
        out = model(idx)
        pred = torch.argmax(out).item()

    return idx2word[pred]

In [103]:
print("machine=>", predict_next("machine"))
print("is=>", predict_next("is"))
print("word2vec=>", predict_next("word2vec"))

machine=> i
is=> is
word2vec=> learning
